In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import pandas as pd

csv_path = "/kaggle/input/notebooks/aabdollahii/persian-wiki-data-knowledge-graph-analysis/cleaned_knowledge_graph.csv"  
df = pd.read_csv(csv_path, encoding="utf-8-sig")

print(df.shape)
df.head()


(1848357, 7)


,Subject,Predicate,Object,subject_len,predicate_len,object_len,object_type
0,سعدی,نام,سعدی شیرازی,4,3,11,text
1,سعدی,زمینه فعالیت,شعر و نثر فارسی,4,12,15,text
2,سعدی,ملیت,ایرانی,4,4,6,text
3,سعدی,تاریخ تولد,بین ۵۸۵ تا ۶۱۵ هجری قمری,4,10,24,text
4,سعدی,محل تولد,شیراز، ایران,4,8,12,text


In [3]:
top_n = 30
predicate_counts = df["Predicate"].value_counts().head(top_n)
predicate_counts


Predicate
استان            86362
شهرستان          82890
بخش              60290
جمعیت            55483
دهستان           54301
نام              24360
شماره ثبت        22834
ملیت             22332
کاربری           22034
زبان             20901
پیشه             19409
دیرینگی          18432
دوره ساخت اثر    17985
نام رسمی         17980
کشور             17644
عنوان            17404
مدت              16455
کارگردان         15867
محل تولد         15493
نوع              15400
مبدا             13949
نویسنده          11853
حضیض             11782
اوج              11735
انحراف           11698
خروج از مرکز     11698
تناوب            11659
آنومالی متوسط    11602
زمینه فعالیت     11552
تاریخ_کشف        11086
Name: count, dtype: int64

In [8]:
# I was worried about the output of the work.

suspicious_preds = [
    "اوج",
    "انحراف",
    "خروج از مرکز",
    "تناوب",
    "آنومالی متوسط",
    "تاریخ_کشف",
]

for p in suspicious_preds:
    print("=" * 80)
    print(f"نمونه‌هایی برای predicate: {p}")
    print("=" * 80)
    # چند ردیف اول با این predicate
    sample_rows = df[df["Predicate"] == p].head(10)
    for idx, row in sample_rows.iterrows():
        subj = row["Subject"]
        obj = row["Object"]
        print(f"Subject: {subj}")
        print(f"Predicate: {p}")
        print(f"Object: {obj}")
        print("-" * 40)


نمونه‌هایی برای predicate: اوج
Subject: زحل
Predicate: اوج
Object: ۱٬۵۰۳٬۵۰۹٬۲۲۹ کیلومتر۱۰٫۰۵ AU
----------------------------------------
Subject: اورانوس
Predicate: اوج
Object: ۳٬۰۰۴٬۴۱۹٬۷۰۴ کیلومتر۲۰٫۰۸۳ ۳۰۵ ۲۶ واحد نجومی
----------------------------------------
Subject: پلوتون
Predicate: اوج
Object: ۷٬۳۱۱٬۰۰۰٬۰۰۰ km۴۸٫۸۷۱ AU
----------------------------------------
Subject: ایستگاه فضایی میر
Predicate: اوج
Object: ۳۹۸ کیلومتر
----------------------------------------
Subject: آپولو ۱
Predicate: اوج
Object: ۳۰۰ کیلومتر (برنامه‌ریزی شده)
----------------------------------------
Subject: وسخود ۲
Predicate: اوج
Object: ۴۷۵ کیلومتر
----------------------------------------
Subject: وستوک۱
Predicate: اوج
Object: ۳۲۷ کیلومتر
----------------------------------------
Subject: سایوز تی‌ام‌ای-۱۲
Predicate: اوج
Object: ۲۳۷ کیلومتر
----------------------------------------
Subject: سایوز تی‌ام‌ای-۹
Predicate: اوج
Object: ۲۴۲ کیلومتر
----------------------------------------
Subject: اسب آبی ۴۲۶
Pred

In [11]:
import random

PREDICATE_TEMPLATES = {
    "استان": [
        "{subj} در استان {obj} قرار دارد.",
        "{subj} یکی از شهرهای استان {obj} است.",
    ],
    "شهرستان": [
        "{subj} در شهرستان {obj} واقع شده است.",
        "{subj} یکی از مناطق شهرستان {obj} است.",
    ],
    "بخش": [
        "{subj} در بخش {obj} قرار دارد.",
        "{subj} در بخش {obj} واقع است.",
    ],
    "دهستان": [
        "{subj} در دهستان {obj} قرار گرفته است.",
        "{subj} در دهستان {obj} واقع شده است.",
    ],
    "کشور": [
        "{subj} در کشور {obj} واقع شده است.",
        "کشور {obj} محل قرارگیری {subj} است.",
    ],

    "جمعیت": [
        "جمعیت {subj} برابر با {obj} نفر است.",
        "تعداد ساکنان {subj} حدود {obj} نفر است.",
    ],
    "دیرینگی": [
        "دیرینگی {subj} به {obj} بازمی‌گردد.",
        "{subj} دارای دیرینگی {obj} است.",
    ],
    "دوره ساخت اثر": [
        "دوره ساخت اثر {subj}، {obj} است.",
        "{subj} در دوره {obj} ساخته شده است.",
    ],

    "نام": [
        "نام {subj}، {obj} است.",
        "{subj} با نام {obj} شناخته می‌شود.",
    ],
    "نام رسمی": [
        "نام رسمی {subj}، {obj} است.",
        "{subj} به طور رسمی {obj} نامیده می‌شود.",
    ],
    "عنوان": [
        "عنوان {subj}، {obj} است.",
        "{subj} با عنوان {obj} شناخته می‌شود.",
    ],
    "شماره ثبت": [
        "شماره ثبت {subj}، {obj} است.",
        "{subj} با شماره ثبت {obj} به ثبت رسیده است.",
    ],

    "ملیت": [
        "ملیت {subj}، {obj} است.",
        "{subj} یک شخصیت {obj} است.",
    ],
    "زبان": [
        "زبان {subj}، {obj} است.",
        "{subj} به زبان {obj} است.",
    ],
    "پیشه": [
        "پیشه {subj}، {obj} است.",
        "{subj} به عنوان {obj} فعالیت می‌کند.",
    ],
    "محل تولد": [
        "{subj} در {obj} به دنیا آمده است.",
        "محل تولد {subj}، {obj} است.",
    ],
    "زمینه فعالیت": [
        "زمینه فعالیت {subj}، {obj} است.",
        "{subj} در زمینه {obj} فعالیت می‌کند.",
    ],

    "کارگردان": [
        "کارگردان {subj}، {obj} است.",
        "{subj} به کارگردانی {obj} ساخته شده است.",
    ],
    "نویسنده": [
        "نویسنده {subj}، {obj} است.",
        "{subj} به قلم {obj} نوشته شده است.",
    ],
    "نوع": [
        "نوع {subj}، {obj} است.",
        "{subj} از نوع {obj} است.",
    ],
    "کاربری": [
        "کاربری {subj}، {obj} است.",
        "{subj} با کاربری {obj} ساخته شده است.",
    ],
    "مدت": [
        "مدت {subj}، {obj} است.",
        "زمان {subj}، {obj} است.",
    ],
    "مبدا": [
        "مبدا {subj}، {obj} است.",
        "{subj} از {obj} آغاز می‌شود.",
    ],

    "حضیض": [
        "حضیض مداری {subj}، {obj} است.",
        "{subj} دارای حضیض {obj} است.",
    ],
    "اوج": [
        "اوج مداری {subj}، {obj} است.",
        "{subj} دارای اوج {obj} است.",
    ],
    "انحراف": [
        "انحراف مداری {subj}، {obj} است.",
        "زاویه انحراف مدار {subj}، {obj} است.",
    ],
    "خروج از مرکز": [
        "خروج از مرکز مدار {subj}، {obj} است.",
        "{subj} دارای خروج از مرکز {obj} است.",
    ],
    "تناوب": [
        "تناوب مداری {subj}، {obj} است.",
        "دوره تناوب {subj}، {obj} است.",
    ],
    "آنومالی متوسط": [
        "آنومالی متوسط {subj}، {obj} است.",
        "مقدار آنومالی متوسط {subj}، {obj} است.",
    ],
    "تاریخ_کشف": [
        "تاریخ کشف {subj}، {obj} است.",
        "{subj} در تاریخ {obj} کشف شده است.",
    ],
}

GENERIC_TEMPLATES = [
    "{pred}ِ {subj}، {obj} است.",
    "{pred} مربوط به {subj}، {obj} است."
]


In [14]:
def verbalize_row(row):
    subj = str(row["Subject"]).strip()
    pred = str(row["Predicate"]).strip()
    obj  = str(row["Object"]).strip()
    
    if pred in PREDICATE_TEMPLATES:
        templates = PREDICATE_TEMPLATES[pred]
        template = random.choice(templates)
    else:
        template = random.choice(GENERIC_TEMPLATES)
    
    sentence = template.format(subj=subj, pred=pred, obj=obj)
    return sentence


In [13]:
#  sanity check
for i in range(10):
    row = df.iloc[i]
    print(row["Subject"], " - ", row["Predicate"], " - ", row["Object"])
    print("=>", verbalize_row(row))
    print("-" * 60)


سعدی  -  نام  -  سعدی شیرازی
=> سعدی با نام سعدی شیرازی شناخته می‌شود.
------------------------------------------------------------
سعدی  -  زمینه فعالیت  -  شعر و نثر فارسی
=> سعدی در زمینه شعر و نثر فارسی فعالیت می‌کند.
------------------------------------------------------------
سعدی  -  ملیت  -  ایرانی
=> سعدی یک شخصیت ایرانی است.
------------------------------------------------------------
سعدی  -  تاریخ تولد  -  بین ۵۸۵ تا ۶۱۵ هجری قمری
=> تاریخ تولد مربوط به سعدی، بین ۵۸۵ تا ۶۱۵ هجری قمری است.
------------------------------------------------------------
سعدی  -  محل تولد  -  شیراز، ایران
=> محل تولد سعدی، شیراز، ایران است.
------------------------------------------------------------
سعدی  -  تاریخ مرگ  -  بین ۶۹۰ تا ۶۹۵
=> تاریخ مرگِ سعدی، بین ۶۹۰ تا ۶۹۵ است.
------------------------------------------------------------
سعدی  -  محل مرگ  -  شیراز، ایران
=> محل مرگِ سعدی، شیراز، ایران است.
------------------------------------------------------------
سعدی  -  محل زندگی  -  شیراز، ب

In [15]:
num_samples = min(100_000, len(df))  # 
subset = df.iloc[:num_samples].copy()

sentences = subset.apply(verbalize_row, axis=1)

output_txt_path = "/kaggle/working/kg_sentences_sample.txt"
with open(output_txt_path, "w", encoding="utf-8") as f:
    for s in sentences:
        f.write(s.strip() + "\n")

print("Sample sentences saved to:", output_txt_path)
print("Example lines:")
for i in range(5):
    print(sentences.iloc[i])


Sample sentences saved to: /kaggle/working/kg_sentences_sample.txt
Example lines:
سعدی با نام سعدی شیرازی شناخته می‌شود.
زمینه فعالیت سعدی، شعر و نثر فارسی است.
ملیت سعدی، ایرانی است.
تاریخ تولد مربوط به سعدی، بین ۵۸۵ تا ۶۱۵ هجری قمری است.
محل تولد سعدی، شیراز، ایران است.


In [17]:
file_path = "/kaggle/working/kg_sentences_sample.txt"

import os
print("Exists:", os.path.exists(file_path))

if os.path.exists(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        n = 100
        for i in range(n):
            line = f.readline()
            if not line:
                break
            print(f"{i+1:02d}: {line.rstrip()}")
else:
    print("File not found:", file_path)


Exists: True
01: سعدی با نام سعدی شیرازی شناخته می‌شود.
02: زمینه فعالیت سعدی، شعر و نثر فارسی است.
03: ملیت سعدی، ایرانی است.
04: تاریخ تولد مربوط به سعدی، بین ۵۸۵ تا ۶۱۵ هجری قمری است.
05: محل تولد سعدی، شیراز، ایران است.
06: تاریخ مرگ مربوط به سعدی، بین ۶۹۰ تا ۶۹۵ است.
07: محل مرگ مربوط به سعدی، شیراز، ایران است.
08: محل زندگیِ سعدی، شیراز، بغداد است.
09: مدفن مربوط به سعدی، سعدیه، شیراز است.
10: در زمان حکومتِ سعدی، اتابکان فارس، عباسیان، خوارزمشاهیان، ایلخانان است.
11: اتفاقات مهم مربوط به سعدی، حملهٔ مغول به ایران است.
12: لقب مربوط به سعدی، افصح‌المتکلمین، استاد سخن، پادشاه سخن، شیخ اجلّ است.
13: دین مربوط به سعدی، اسلام، سنی است.
14: مذهبِ سعدی، شافعی، اشعری است.
15: سبک نوشتاری مربوط به سعدی، سبک عراقی است.
16: کتاب‌ها مربوط به سعدی، ''گلستان''، ''بوستان''، ''غزلیات سعدی'' است.
17: دیوان اشعارِ سعدی، ''کلّیات سعدی'' است.
18: تحصیلات مربوط به سعدی، دانش‌آموختهٔ مدرسهٔ نظامیهٔ بغداد است.
19: شاگرد مربوط به سعدی، سِبط بن جوزی، شهاب‌الدّینْ عُمَرِ سهروردی است.
20: تأثیرگذاشته برِ 

In [19]:
# اعمال verbalize_row روی تمام ردیف‌های دیتافریم (بدون محدودیت تعداد)
sentences = df.apply(verbalize_row, axis=1)

# ذخیره تمامی جملات در یک فایل txt
output_txt_path = "/kaggle/working/kg_sentences_all.txt"
with open(output_txt_path, "w", encoding="utf-8") as f:
    for s in sentences:
        f.write(str(s).strip() + "\n")

print("All sentences saved to:", output_txt_path)
print("Total sentences written:", len(sentences))

# نمایش ۵ خط اول برای اطمینان
print("Example lines:")
for i in range(min(5, len(sentences))):
    print(sentences.iloc[i])


All sentences saved to: /kaggle/working/kg_sentences_all.txt
Total sentences written: 1848357
Example lines:
سعدی با نام سعدی شیرازی شناخته می‌شود.
زمینه فعالیت سعدی، شعر و نثر فارسی است.
ملیت سعدی، ایرانی است.
تاریخ تولد مربوط به سعدی، بین ۵۸۵ تا ۶۱۵ هجری قمری است.
محل تولد سعدی، شیراز، ایران است.
